In [1]:
import numpy as np 
import time

import sys
sys.path.append('../')
from reachy import Reachy, parts
from behavior.manipulate_flyer import Manipulate_flyer

In [2]:
reachy = Reachy(
    head=parts.Head(io='/dev/ttyUSB*'),
    right_arm=parts.RightArm(io='/dev/ttyUSB*', hand='force_gripper'),
    left_arm=parts.LeftArm(io='/dev/ttyUSB*', hand='flyer_hand')
)

In [1]:
from pyluos import Device

In [2]:
d = Device('/dev/ttyUSB0')

In [3]:
d.modules

-------------------------------------------------
Type                Alias               ID   
-------------------------------------------------
Gate                r_right_arm         1    
DynamixelMotor      dxl_10              2    
DynamixelMotor      dxl_11              3    
DynamixelMotor      dxl_12              4    
DynamixelMotor      dxl_13              5    
DynamixelMotor      dxl_14              6    
DynamixelMotor      dxl_15              7    
DynamixelMotor      dxl_16              8    
DynamixelMotor      dxl_17              9    
Load                force_gripper       10   

In [5]:
d.dxl_17.dxl_detect()

To get new detected Dxl motors usable on pyluos you should recreate your Pyluos object.


In [4]:
d.dxl_17.rot_position

3.079

# Left arm

In [3]:
current_position = [m.present_position for m in reachy.left_arm.motors]
current_position

[9.647999999999996, 17.076999999999998, -47.341, -50.681, 15.396, -57.978]

In [56]:
goal_position_1 = [-8.11, 41.165, -75.121, -88.571, 26.54, -42]
#goal_position_2 = [-6.3, 44.154, -73.363, -88.308, 24.487, 46.198]
goal_position = [9.120999999999995, 34.571, -59.209, -90.066, 11.877, -40]
#base_pos_left = [-10.9, 13.6, -73.978, -35.209, 35.924, -55]
base_pos_left = [10.4, 17.1, -45.934, -51.648, 12.757, -55.253]
#base_pos_left = [8.857, 18.57, -43.033, -49.802, 18.915, 60.176]

In [57]:
for m in reachy.left_arm.motors:
    m.compliant = False

In [58]:
reachy.goto({
        m.name: j
        for j, m in zip(goal_position, reachy.left_arm.motors)
    }, duration=2, wait=True, interpolation_mode='minjerk')

In [59]:
for m in reachy.left_arm.motors:
    m.compliant = True

In [31]:
reachy.left_arm.elbow_pitch.temperature

44.0

module wrist_fan have been excluded from the network due to no responses.
module wrist_fan have been excluded from the network due to no responses.


# Right arm

In [4]:
current_position = [m.present_position for m in reachy.right_arm.motors]
current_position

[5.076999999999998,
 -5.120999999999995,
 30.286,
 -56.308,
 -100.733,
 -6.286,
 37.683,
 16.276]

In [53]:
right_arm_step1 = [-0.5, -56.549, 52.44, -117.319, -106.012, 28.527, -13.343, -23.607]
#[1.3850000000000051, -23.495000000000005, 55.341, -112.22, -105.132, 27.912, -13.343, -23.021]

right_arm_step2 = [-5.6, -40.901, 62.022, -114.593, -108.944, 31.78, -24.487, -48.827]
right_arm_step21 = [-3.27, -33.429, 60.879, -111.165, -105.425, 28.527, -26.246, -47.361]
right_arm_step22 = [-3.45, -35.363, 61.055, -110.11, -103.666, 29.582, -25.66, -47.361]
right_arm_step23 = [-1.95, -34.659, 58.945, -120.484, -107.478, 30.813, -33.578, -47.067]
right_arm_step24 = [-0.54, -33.78, 57.538, -115.121, -103.959, 30.637, -32.405, -45.601]

base_pos_right = [13.4, -17.78, 31.692, -67.121, -101.906, -1.275, 33.871, 18.328]


A = reachy.right_arm.forward_kinematics(joints_position=right_arm_step2)
A[2][3] -= 0.05

JA = reachy.right_arm.inverse_kinematics(A,q0=right_arm_step2)
print(np.round(JA,2))

B = A.copy()
B[1][3] -= 0.05

JB = reachy.right_arm.inverse_kinematics(B,q0=right_arm_step2)
print(np.round(JB,2))

C = B.copy()
C[1][3] -= 0.1

JC = reachy.right_arm.inverse_kinematics(C,q0=right_arm_step2)
JC[7] = 90
print(np.round(JC,2))

test = [57.4, -45.912, 25.89, -126.637, -80.792, 9.011, 15.689, 17.449]

[  -1.95  -37.07   61.63 -105.03 -100.     29.44  -18.52  -48.83]
[  -0.63  -44.72   58.98 -105.54 -100.     30.96  -11.84  -48.83]
[   3.18  -62.86   54.58 -105.22  -98.53   35.54   11.07   90.  ]


In [60]:
for m in reachy.right_arm.motors:
    m.compliant = False

In [55]:
reachy.goto({
        m.name: j
        for j, m in zip(right_arm_step24, reachy.right_arm.motors)
    }, duration=2, wait=True, interpolation_mode='minjerk')


In [57]:
reachy.goto({
        m.name: j
        for j, m in zip(right_arm_step1, reachy.right_arm.motors)
    }, duration=2, wait=True, interpolation_mode='minjerk')

reachy.goto({
        m.name: j
        for j, m in zip(right_arm_step2, reachy.right_arm.motors)
    }, duration=1, wait=True, interpolation_mode='minjerk')

reachy.goto({
        m.name: j
        for j, m in zip(JA, reachy.right_arm.motors)
    }, duration=1, wait=True, interpolation_mode='minjerk')

reachy.goto({
        m.name: j
        for j, m in zip(JB, reachy.right_arm.motors)
    }, duration=1, wait=True, interpolation_mode='minjerk')

reachy.goto({
    'right_arm.hand.gripper': 90,
}, duration = 1, wait=True)

reachy.goto({
        m.name: j
        for j, m in zip(JC, reachy.right_arm.motors)
    }, duration=2, wait=True, interpolation_mode='minjerk')

In [50]:
reachy.goto({        
        'right_arm.shoulder_pitch': -25,
        'right_arm.shoulder_roll': -10,
        'right_arm.arm_yaw': 24,    
        'right_arm.elbow_pitch': -100,
        'right_arm.hand.forearm_yaw': -140,
        'right_arm.hand.wrist_pitch': 0,
        'right_arm.hand.wrist_roll': -40,
        }, duration=2,wait=True,starting_point='goal_position', interpolation_mode='minjerk'),

([<reachy.trajectory.interpolation.MinimumJerk at 0xa25d3630>,
  <reachy.trajectory.interpolation.MinimumJerk at 0xb34fd750>],)

In [61]:
for m in reachy.right_arm.motors:
    m.compliant = True

# Test whole movement

In [3]:
goal_position = [9.1, 34.571, -59.209, -90.066, 11.877, -38]
base_pos_left = [10.4, 17.1, -45.934, -51.648, 12.757, -58]

In [4]:
right_arm_step1 = [-0.5, -56.549, 52.44, -117.319, -106.012, 28.527, -13.343, -23.607]

right_arm_step2 = [-0.54, -33.78, 57.538, -115.121, -103.959, 30.637, -32.405, -45.601]

base_pos_right = [13.4, -17.78, 31.692, -67.121, -101.906, -1.275, 33.871, 18.328]

A = reachy.right_arm.forward_kinematics(joints_position=right_arm_step2)
A[2][3] -= 0.05

JA = reachy.right_arm.inverse_kinematics(A,q0=right_arm_step2)
print(np.round(JA,2))

B = A.copy()
B[1][3] -= 0.06

JB = reachy.right_arm.inverse_kinematics(B,q0=right_arm_step2)
print(np.round(JB,2))

C = B.copy()
#C[1][3] -= 0.1
C[2][3] +=0.1

JC = reachy.right_arm.inverse_kinematics(C,q0=right_arm_step2)
JC[7] = 90
print(np.round(JC,2))

test = [-5.5, -71.407, 55.868, -114.945, -132.111, 32.747, -25.073, 19.208]

[  -0.45  -30.71   59.29 -104.15  -96.47   32.11  -24.02  -45.6 ]
[   2.6   -43.34   54.67 -107.87 -100.     34.75  -15.67  -45.6 ]
[   9.72  -45.18   42.73 -125.   -100.     37.49  -22.26   90.  ]


In [5]:
for m in reachy.right_arm.motors:
    m.compliant = False
for m in reachy.left_arm.motors:
    m.use_static_error_fix = True
    m.compliant = False

In [7]:
reachy.goto({
        m.name: j
        for j, m in zip(goal_position, reachy.left_arm.motors)
    }, duration=1, wait=False, interpolation_mode='minjerk')

time.sleep(0.2)

reachy.goto({
        m.name: j
        for j, m in zip(right_arm_step1, reachy.right_arm.motors)
    }, duration=1, wait=True, interpolation_mode='minjerk')

reachy.goto({
        m.name: j
        for j, m in zip(right_arm_step2, reachy.right_arm.motors)
    }, duration=0.5, wait=True)#, interpolation_mode='minjerk')

reachy.goto({
        m.name: j
        for j, m in zip(JA, reachy.right_arm.motors)
    }, duration=0.5, wait=True)#, interpolation_mode='minjerk')

reachy.goto({
        m.name: j
        for j, m in zip(JB, reachy.right_arm.motors)
    }, duration=0.5, wait=True)#, interpolation_mode='minjerk')

reachy.goto({
    'right_arm.hand.gripper': 90,
}, duration = 0.5, wait=True)

reachy.goto({
        m.name: j
        for j, m in zip(JC, reachy.right_arm.motors)
    }, duration=0.7, wait=False, interpolation_mode='minjerk')

time.sleep(0.3)

reachy.goto({
        m.name: j
        for j, m in zip(base_pos_left, reachy.left_arm.motors)
    }, duration=1, wait=False, interpolation_mode='minjerk')

time.sleep(0.7)

reachy.goto({        
        'right_arm.shoulder_pitch': -25,
        'right_arm.shoulder_roll': -10,
        'right_arm.arm_yaw': 24,    
        'right_arm.elbow_pitch': -100,
        'right_arm.hand.forearm_yaw': -140,
        'right_arm.hand.wrist_pitch': 0,
        'right_arm.hand.wrist_roll': -40,
        }, duration=2,wait=True,starting_point='goal_position', interpolation_mode='minjerk'),


reachy.goto({
    'right_arm.hand.gripper': 0,
}, duration = 0.5, wait=True)

time.sleep(1)

reachy.goto({
        m.name: j
        for j, m in zip(base_pos_right, reachy.right_arm.motors)
    }, duration=1.2, wait=True, interpolation_mode='minjerk')

In [8]:
for m in reachy.right_arm.motors:
    m.compliant = True
for m in reachy.left_arm.motors:
    m.compliant = True

In [12]:
reachy.head.compliant = False

In [134]:
reachy.head.look_at(0.5,0.1,-0.4, duration=2,wait=True)

In [135]:
reachy.head.compliant = True

# Test integration

In [3]:
manip = Manipulate_flyer(reachy)

In [4]:
reachy.head.compliant = False

for m in reachy.head.motors:
    m.compliant = False


In [5]:
manip.play('grab_flyer')
manip.play('pull_flyer_adapted')
manip.play('hold_flyer_adapted')
manip.play('give_flyer_adapted')

essai :  91.294


essai :  68.948


In [25]:
for m in reachy.left_arm.motors:
    m.compliant = True

In [6]:
reachy.head.compliant = True

for m in reachy.head.motors:
    m.compliant = True

In [14]:
import behavior.flyer_actions as fa

In [9]:
for m in reachy.right_arm.motors:
    m.compliant = True

In [8]:
reachy.right_arm.hand.gripper.goal_position = 50
time.sleep(0.1)
reachy.right_arm.hand.gripper.goal_position = 0
time.sleep(0.1)
reachy.right_arm.hand.gripper.goal_position = 50

In [21]:
t = fa.initialize_gripper_threshold(reachy)
print('seuil : ',t)
fa.grab_flyer(reachy)
fa.pull_flyer_adapted(reachy,t,0.5,0,0)
manip.play('hold_flyer_adapted')
manip.play('give_flyer_adapted')

seuil :  70.65533333333333
essai :  74.46


In [28]:
for m in reachy.right_arm.motors:
    m.compliant = False

In [29]:
reachy.right_arm.hand.gripper.goal_position = 50
time.sleep(0.5)

r = reachy.right_arm.hand.grip_force
print(r)

80.044


In [30]:
for m in reachy.right_arm.motors:
    m.compliant = True

In [44]:
reachy.right_arm.motors[:7]

[<DxlMotor "right_arm.shoulder_pitch" pos="10.0" mode="stiff">,
 <DxlMotor "right_arm.shoulder_roll" pos="-9.956000000000003" mode="stiff">,
 <DxlMotor "right_arm.arm_yaw" pos="29.231" mode="stiff">,
 <DxlMotor "right_arm.elbow_pitch" pos="-43.033" mode="stiff">,
 <DxlMotor "right_arm.hand.forearm_yaw" pos="-11.877" mode="stiff">,
 <DxlMotor "right_arm.hand.wrist_pitch" pos="-60.703" mode="stiff">,
 <DxlMotor "right_arm.hand.wrist_roll" pos="-15.396" mode="stiff">]